# 01 · The modelling tables

What the models learn from. Both tables are built by dbt and exported with
`python -m chama.pipeline export`; every feature uses only what was known at the
time (the loan's issue month, or the meeting).

| Target | One row per | Label | Events |
| --- | --- | --- | --- |
| `loan_default` | loan (`loan_features`) | `defaulted`: rolled over, overdue or lost track | about 1 in 9 |
| `missed_repayment` | member per meeting while a loan runs (`member_months`) | `missed_ahead`: a repayment missed in the next 3 meetings | about 1 in 5 |

> The tables hold member codes and features only: no names, no free text (decision D-016). The loan-default label is **provisional** until the Chama agrees it (D-018): `defaulted` = rolled over while unpaid, overdue (confirmed), or lost track.

In [ ]:
# Setup: runs in Google Colab and on this computer.
# Colab, once: add a GitHub token that can read this repository as the Colab
# secret GITHUB_TOKEN (key icon on the left). This cell mounts Drive, fetches
# the code and installs what Colab lacks. On this computer it uses DRIVE_DIR
# from .env and the `chama` conda environment.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "m2-features-models"

if IN_COLAB:
    from google.colab import drive, userdata

    REPO = "/content/chama-loan-default"
    OWNER = "muriithi-alvin"
    PLAIN_URL = f"https://github.com/{OWNER}/chama-loan-default.git"

    def git(*args):
        """Run git with the token; show git's message, never the token."""
        token = userdata.get("GITHUB_TOKEN").strip()
        url = PLAIN_URL.replace("https://", f"https://{OWNER}:{token}@")
        result = subprocess.run(["git", *[url if a == "URL" else a for a in args]],
                                capture_output=True, text=True)
        if result.returncode:
            message = (result.stderr or result.stdout).replace(token, "***").strip()
            raise RuntimeError(f"git failed: {message}")

    drive.mount("/content/drive")
    if os.path.exists(REPO):
        git("-C", REPO, "pull", "-q", "URL", BRANCH)
    else:
        git("clone", "-q", "--branch", BRANCH, "URL", REPO)
        git("-C", REPO, "remote", "set-url", "origin", PLAIN_URL)   # no token left behind
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlflow", "shap", "imbalanced-learn", "python-dotenv"], check=True)
    sys.path.insert(0, f"{REPO}/src")
    os.environ["DRIVE_DIR"] = "/content/drive/MyDrive/chama-loan-default"


import warnings

import pandas as pd

from chama.config import export_dir, load_config, mlflow_dir
from chama.models.data import TARGETS, known_outcomes, load_table, xy
from chama.models.train import target_settings

# the plots are matplotlib Figures built without pyplot: show them as images
%matplotlib inline
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
CONFIG = load_config()
DATA_DIR = export_dir()
print("data:  ", DATA_DIR)
print("MLflow:", mlflow_dir())

## Loans (`loan_default`)

In [ ]:
from chama.models.data import LOAN_DEFAULT, MISSED_REPAYMENT, load_manifest

loans = load_table(DATA_DIR / "loan_features.parquet", LOAN_DEFAULT)
manifest = load_manifest(DATA_DIR / "loan_features.parquet")
{k: manifest.get(k) for k in ["exported_at", "rows", "training_rows", "events",
                              "members", "default_statuses", "git_commit"]}

In [ ]:
def by_year(df, target):
    known = known_outcomes(df)
    table = known.groupby(known[target.time].dt.year).agg(
        rows=(target.label, "size"), events=(target.label, "sum"))
    table["event_rate"] = (table["events"] / table["rows"]).round(2)
    table["tested"] = table.index.isin(target_settings(CONFIG, target.name)["test_years"])
    return table


by_year(loans, LOAN_DEFAULT)

In [ ]:
known_loans = known_outcomes(loans)
by_type = known_loans.groupby("loan_type").agg(loans=("defaulted", "size"),
                                                defaults=("defaulted", "sum"))
by_type["default_rate"] = (by_type["defaults"] / by_type["loans"]).round(2)
by_type

Empty is meaningful: `amount_vs_own_max` is empty for a member's first loan of a
type, `recent_contribution_rate` before contributions were written down (2017). The
models fill gaps with the training median and add a column saying the value was
missing. Money appears only as ratios, because loan sizes grew about six-fold.

In [ ]:
def compare(df, target):
    known = known_outcomes(df)
    numeric = [c for c in target.features if c not in target.categorical_features]
    table = known.groupby(target.label)[numeric].mean().T.round(2)
    table.columns = ["no event (0)", "event (1)"]
    table["missing"] = known[numeric].isna().mean().round(2)
    return table


compare(loans, LOAN_DEFAULT)

## Members per meeting (`missed_repayment`)

One row per member per meeting while the member owes on a loan. The label looks at
the next 3 meetings, so the last 3 meetings of the records have no label yet.

In [ ]:
months = load_table(DATA_DIR / "member_months.parquet", MISSED_REPAYMENT)
by_year(months, MISSED_REPAYMENT)

In [ ]:
compare(months, MISSED_REPAYMENT)